# HPRC Ensembl vs CAT Annotation QC Report

This notebook generates a comprehensive QC report comparing Ensembl (linear projection) and CAT (graph-based projection) gene annotations across 400+ HPRC assemblies.

**Memory-efficient design**: Processes data in chunks, computes summaries, and saves to disk immediately.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import gc
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Configuration

In [ ]:
# Set paths
OUTPUT_DIR = Path('../results')  # Adjust this path as needed
QC_DIR = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR = OUTPUT_DIR / 'results'
SUMMARY_DIR = OUTPUT_DIR / 'summary_stats'  # Where we'll save intermediate results
SUMMARY_DIR.mkdir(exist_ok=True, parents=True)

# Processing parameters
CHUNK_SIZE = 50  # Process 50 assemblies at a time

print(f"Output directory: {OUTPUT_DIR}")
print(f"QC metrics directory: {QC_DIR}")
print(f"Summary stats directory: {SUMMARY_DIR}")
print(f"Chunk size: {CHUNK_SIZE} assemblies")

## Gencode Version Filtering (Optional)

CAT and Ensembl may have used different Gencode versions as their reference. To enable fair comparison, we can filter to "stable" genes/transcripts that exist in both versions.

**To enable this filtering:**
1. Set the paths to the Gencode GTF files used by each annotation method
2. Run the cells below to create stable gene sets
3. Filtered analyses will be generated alongside the full analyses

In [ ]:
# Gencode version configuration
# Set these paths to enable Gencode version filtering
# Leave as None to skip filtering

GENCODE_CAT_GTF = None  # e.g., Path('/path/to/gencode.v47.annotation.gtf.gz')
GENCODE_ENSEMBL_GTF = None  # e.g., Path('/path/to/gencode.v44.annotation.gtf.gz')

# Enable filtering only if both paths are provided
ENABLE_GENCODE_FILTERING = GENCODE_CAT_GTF is not None and GENCODE_ENSEMBL_GTF is not None

if ENABLE_GENCODE_FILTERING:
    print(f"Gencode filtering ENABLED")
    print(f"  CAT Gencode: {GENCODE_CAT_GTF}")
    print(f"  Ensembl Gencode: {GENCODE_ENSEMBL_GTF}")
else:
    print("Gencode filtering DISABLED (set GENCODE_CAT_GTF and GENCODE_ENSEMBL_GTF to enable)")

In [ ]:
import gzip
from collections import defaultdict

def parse_gencode_gtf(gtf_path):
    """
    Parse a Gencode GTF file and extract gene/transcript information.
    
    Returns:
        genes: dict {gene_id: {'name': str, 'biotype': str, 'chrom': str}}
        transcripts: dict {transcript_id: {'gene_id': str, 'exons': [(start, end), ...]}}
    """
    genes = {}
    transcripts = defaultdict(lambda: {'gene_id': None, 'exons': []})
    
    opener = gzip.open if str(gtf_path).endswith('.gz') else open
    
    with opener(gtf_path, 'rt') as f:
        for line in f:
            if line.startswith('#'):
                continue
            
            parts = line.strip().split('\t')
            if len(parts) < 9:
                continue
            
            chrom, source, feature, start, end, score, strand, frame, attrs = parts
            
            # Parse attributes
            attr_dict = {}
            for attr in attrs.split(';'):
                attr = attr.strip()
                if ' ' in attr:
                    key, val = attr.split(' ', 1)
                    attr_dict[key] = val.strip('"')
            
            gene_id = attr_dict.get('gene_id', '').split('.')[0]  # Remove version
            
            if feature == 'gene':
                genes[gene_id] = {
                    'name': attr_dict.get('gene_name', ''),
                    'biotype': attr_dict.get('gene_type', attr_dict.get('gene_biotype', '')),
                    'chrom': chrom
                }
            
            elif feature == 'transcript':
                tx_id = attr_dict.get('transcript_id', '').split('.')[0]
                transcripts[tx_id]['gene_id'] = gene_id
            
            elif feature == 'exon':
                tx_id = attr_dict.get('transcript_id', '').split('.')[0]
                transcripts[tx_id]['exons'].append((int(start), int(end)))
    
    # Sort exons for each transcript
    for tx_id in transcripts:
        transcripts[tx_id]['exons'].sort()
    
    return genes, dict(transcripts)


def create_stable_gene_set(genes_v1, genes_v2):
    """
    Create a set of gene IDs present in both Gencode versions.
    """
    return set(genes_v1.keys()) & set(genes_v2.keys())


def create_stable_transcript_set(tx_v1, tx_v2):
    """
    Create a set of transcript IDs with identical exon structure in both versions.
    """
    stable = set()
    
    common_tx = set(tx_v1.keys()) & set(tx_v2.keys())
    
    for tx_id in common_tx:
        exons_1 = tuple(tx_v1[tx_id]['exons'])
        exons_2 = tuple(tx_v2[tx_id]['exons'])
        
        if exons_1 == exons_2:
            stable.add(tx_id)
    
    return stable


# Load Gencode data if filtering is enabled
stable_genes = None
stable_transcripts = None

if ENABLE_GENCODE_FILTERING:
    print("Loading Gencode GTF files...")
    
    print(f"  Parsing CAT Gencode: {GENCODE_CAT_GTF}")
    cat_genes, cat_transcripts = parse_gencode_gtf(GENCODE_CAT_GTF)
    print(f"    Found {len(cat_genes):,} genes, {len(cat_transcripts):,} transcripts")
    
    print(f"  Parsing Ensembl Gencode: {GENCODE_ENSEMBL_GTF}")
    ens_genes, ens_transcripts = parse_gencode_gtf(GENCODE_ENSEMBL_GTF)
    print(f"    Found {len(ens_genes):,} genes, {len(ens_transcripts):,} transcripts")
    
    print("\nCreating stable sets...")
    stable_genes = create_stable_gene_set(cat_genes, ens_genes)
    print(f"  Stable genes (in both versions): {len(stable_genes):,}")
    
    stable_transcripts = create_stable_transcript_set(cat_transcripts, ens_transcripts)
    print(f"  Stable transcripts (identical exons): {len(stable_transcripts):,}")
    
    # Save stable sets for reference
    pd.DataFrame({'gene_id': list(stable_genes)}).to_csv(
        SUMMARY_DIR / 'stable_genes.tsv', sep='\t', index=False
    )
    pd.DataFrame({'transcript_id': list(stable_transcripts)}).to_csv(
        SUMMARY_DIR / 'stable_transcripts.tsv', sep='\t', index=False
    )
    print(f"\n✓ Stable sets saved to {SUMMARY_DIR}")
    
    # Clean up memory
    del cat_genes, cat_transcripts, ens_genes, ens_transcripts
    gc.collect()
else:
    print("Skipping Gencode filtering (not configured)")

In [ ]:
def filter_to_stable_genes(df, gene_id_col='ensembl_gene_id', stable_set=None):
    """
    Filter a DataFrame to only include rows where the gene is in the stable set.
    
    Args:
        df: DataFrame with gene IDs
        gene_id_col: Column name containing gene IDs (will strip version suffix)
        stable_set: Set of stable gene IDs (without version)
    
    Returns:
        Filtered DataFrame
    """
    if stable_set is None:
        return df
    
    # Strip version suffix from gene IDs for matching
    gene_ids_stripped = df[gene_id_col].str.split('.').str[0]
    mask = gene_ids_stripped.isin(stable_set)
    
    return df[mask].copy()


def compute_filtered_stats(description, df_full, df_filtered, metric_col=None):
    """
    Print comparison statistics between full and filtered datasets.
    """
    n_full = len(df_full)
    n_filtered = len(df_filtered)
    pct_retained = n_filtered / n_full * 100 if n_full > 0 else 0
    
    print(f"\n{description}:")
    print(f"  Full dataset: {n_full:,} records")
    print(f"  Stable genes only: {n_filtered:,} records ({pct_retained:.1f}% retained)")
    
    if metric_col and metric_col in df_full.columns:
        mean_full = df_full[metric_col].mean()
        mean_filt = df_filtered[metric_col].mean() if len(df_filtered) > 0 else 0
        print(f"  Mean {metric_col}: {mean_full:.3f} (full) vs {mean_filt:.3f} (stable)")

print("✓ Filtering helper functions defined")

## 1. Process Transcript Concordance (Chunked)

Process files in chunks to avoid memory issues.

In [ ]:
def process_transcript_concordance_chunked():
    """Process transcript concordance in chunks and save summary stats."""
    files = list(QC_DIR.rglob('*_transcript_concordance.tsv'))
    print(f"Found {len(files)} transcript concordance files")
    
    if not files:
        return None
    
    # Process in chunks
    per_assembly_stats = []
    concordance_rates = []
    jaccard_indices = []
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1} ({len(chunk_files)} files)")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            # Extract rates for overall distribution
            concordance_rates.extend(chunk_data['transcript_concordance_rate'].tolist())
            if 'avg_jaccard_index' in chunk_data.columns:
                jaccard_indices.extend(chunk_data['avg_jaccard_index'].tolist())
            
            # Per-assembly summary
            agg_dict = {
                'transcript_concordance_rate': 'mean',
                'n_exact_matches': 'sum',
                'n_unmatched': 'sum',
                'ensembl_gene_id': 'count'
            }

            # Add columns if present
            for col in ['avg_jaccard_index', 'n_intron_matches', 'n_subset',
                       'n_superset', 'n_partial_5', 'n_partial_3', 'n_other_partial']:
                if col in chunk_data.columns:
                    agg_dict[col] = 'mean' if 'index' in col else 'sum'
            
            assembly_summary = chunk_data.groupby('assembly_accession').agg(agg_dict).reset_index()
            per_assembly_stats.append(assembly_summary)
            
            del chunk_data, chunk_dfs
            gc.collect()
    
    # Combine and save per-assembly stats
    if per_assembly_stats:
        per_assembly_df = pd.concat(per_assembly_stats, ignore_index=True)
        per_assembly_df.to_csv(SUMMARY_DIR / 'transcript_concordance_per_assembly.tsv', 
                               sep='\t', index=False)
        print(f"Saved per-assembly stats: {len(per_assembly_df)} assemblies")
    
    # Save concordance rate distribution
    concordance_df = pd.DataFrame({'concordance_rate': concordance_rates})
    concordance_df.to_csv(SUMMARY_DIR / 'transcript_concordance_rates.tsv', 
                          sep='\t', index=False)
                          
    if jaccard_indices:
        pd.DataFrame({'jaccard_index': jaccard_indices}).to_csv(
            SUMMARY_DIR / 'transcript_jaccard_indices.tsv', sep='\t', index=False
        )
    
    print(f"Saved concordance rates: {len(concordance_rates)} gene pairs")
    
    del concordance_rates, per_assembly_stats, jaccard_indices
    gc.collect()
    
    return per_assembly_df

transcript_summary = process_transcript_concordance_chunked()
print("\n✓ Transcript concordance processing complete")

## 2. Process Coding Integrity (Chunked)

In [ ]:
def process_coding_integrity_chunked():
    """Process coding integrity in chunks and save summary stats."""
    files = list(QC_DIR.rglob('*_coding_integrity.tsv'))
    print(f"Found {len(files)} coding integrity files")
    
    if not files:
        return None
    
    # Accumulators
    total_genes = 0
    genes_with_cds = 0
    start_matches = 0
    stop_matches = 0
    both_matches = 0
    frameshifts = 0
    classification_counts = {}
    length_diffs = []
    per_assembly_stats = []
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                # Convert boolean strings
                for col in ['start_codon_match', 'stop_codon_match', 'frameshift_detected', 
                           'has_ensembl_cds', 'has_cat_cds']:
                    if col in df.columns:
                        df[col] = df[col].map({'True': True, 'False': False, True: True, False: False})
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            # Filter to genes with CDS in both
            with_cds = chunk_data[
                (chunk_data['has_ensembl_cds'] == True) & 
                (chunk_data['has_cat_cds'] == True)
            ]
            
            # Accumulate basic stats
            total_genes += len(chunk_data)
            genes_with_cds += len(with_cds)
            
            if len(with_cds) > 0:
                start_matches += (with_cds['start_codon_match'] == True).sum()
                stop_matches += (with_cds['stop_codon_match'] == True).sum()
                both_matches += ((with_cds['start_codon_match'] == True) & 
                                (with_cds['stop_codon_match'] == True)).sum()
                frameshifts += (with_cds['frameshift_detected'] == True).sum()
                
                # Accumulate classifications
                if 'classification' in with_cds.columns:
                    for cls, count in with_cds['classification'].value_counts().items():
                        classification_counts[cls] = classification_counts.get(cls, 0) + count
                
                # Sample length differences
                length_diffs.extend(with_cds['length_difference'].sample(
                    min(1000, len(with_cds))
                ).tolist())
                
                # Per-assembly stats
                assembly_stats = with_cds.groupby('assembly_accession').apply(
                    lambda x: pd.Series({
                        'n_protein_coding': len(x),
                        'start_stop_agreement': ((x['start_codon_match'] == True) & 
                                                (x['stop_codon_match'] == True)).sum() / len(x),
                        'n_frameshifts': (x['frameshift_detected'] == True).sum(),
                        'n_full_match': (x['classification'] == 'Full_Match').sum() if 'classification' in x.columns else 0
                    })
                ).reset_index()
                per_assembly_stats.append(assembly_stats)
            
            del chunk_data, chunk_dfs, with_cds
            gc.collect()
    
    # Save overall stats
    overall_data = {
        'total_genes': total_genes,
        'genes_with_cds_both': genes_with_cds,
        'start_codon_matches': start_matches,
        'stop_codon_matches': stop_matches,
        'both_codons_match': both_matches,
        'frameshifts_detected': frameshifts,
        'pct_with_cds': genes_with_cds / total_genes if total_genes > 0 else 0,
        'pct_start_match': start_matches / genes_with_cds if genes_with_cds > 0 else 0,
        'pct_stop_match': stop_matches / genes_with_cds if genes_with_cds > 0 else 0,
        'pct_both_match': both_matches / genes_with_cds if genes_with_cds > 0 else 0,
        'pct_frameshift': frameshifts / genes_with_cds if genes_with_cds > 0 else 0
    }
    overall_stats = pd.DataFrame([overall_data])
    overall_stats.to_csv(SUMMARY_DIR / 'coding_integrity_overall.tsv', sep='\t', index=False)
    
    # Save classification counts
    pd.DataFrame([
        {'classification': k, 'count': v, 'percentage': v/genes_with_cds if genes_with_cds > 0 else 0}
        for k, v in classification_counts.items()
    ]).to_csv(SUMMARY_DIR / 'coding_classifications.tsv', sep='\t', index=False)
    
    # Save length differences sample
    pd.DataFrame({'length_difference': length_diffs}).to_csv(
        SUMMARY_DIR / 'cds_length_differences_sample.tsv', sep='\t', index=False
    )
    
    # Save per-assembly stats
    if per_assembly_stats:
        per_assembly_df = pd.concat(per_assembly_stats, ignore_index=True)
        per_assembly_df.to_csv(SUMMARY_DIR / 'coding_integrity_per_assembly.tsv', 
                               sep='\t', index=False)
    
    print(f"Saved coding integrity stats: {total_genes} total genes, {genes_with_cds} with CDS")
    
    return overall_stats

coding_summary = process_coding_integrity_chunked()
print("\n✓ Coding integrity processing complete")

## 3. Process Gene Presence/Absence (Chunked)

In [ ]:
def process_gene_presence_chunked():
    """Process gene presence in chunks and save summary stats."""
    files = list(QC_DIR.rglob('*_gene_presence.tsv'))
    print(f"Found {len(files)} gene presence files")
    
    if not files:
        return None
    
    # Count genes across all assemblies
    gene_counts = {'both': 0, 'ensembl_only': 0, 'cat_only': 0}
    ensembl_missing_genes = {}
    cat_missing_genes = {}
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                # Convert boolean strings
                df['present_in_ensembl'] = df['present_in_ensembl'].map(
                    {'True': True, 'False': False, True: True, False: False}
                )
                df['present_in_cat'] = df['present_in_cat'].map(
                    {'True': True, 'False': False, True: True, False: False}
                )
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            # Count by category
            both = ((chunk_data['present_in_ensembl'] == True) & 
                   (chunk_data['present_in_cat'] == True)).sum()
            ens_only = ((chunk_data['present_in_ensembl'] == True) & 
                       (chunk_data['present_in_cat'] == False)).sum()
            cat_only = ((chunk_data['present_in_ensembl'] == False) & 
                       (chunk_data['present_in_cat'] == True)).sum()
            
            gene_counts['both'] += both
            gene_counts['ensembl_only'] += ens_only
            gene_counts['cat_only'] += cat_only
            
            # Track top missing genes
            ens_missing = chunk_data[
                (chunk_data['present_in_ensembl'] == True) & 
                (chunk_data['present_in_cat'] == False)
            ]
            for gene in ens_missing['gene_name'].value_counts().head(50).index:
                ensembl_missing_genes[gene] = ensembl_missing_genes.get(gene, 0) + \
                    (ens_missing['gene_name'] == gene).sum()
            
            cat_missing = chunk_data[
                (chunk_data['present_in_ensembl'] == False) & 
                (chunk_data['present_in_cat'] == True)
            ]
            for gene in cat_missing['gene_name'].value_counts().head(50).index:
                cat_missing_genes[gene] = cat_missing_genes.get(gene, 0) + \
                    (cat_missing['gene_name'] == gene).sum()
            
            del chunk_data, chunk_dfs
            gc.collect()
    
    # Save summary
    summary = pd.DataFrame([gene_counts])
    total = sum(gene_counts.values())
    summary['total'] = total
    summary['pct_both'] = gene_counts['both'] / total if total > 0 else 0
    summary['pct_ensembl_only'] = gene_counts['ensembl_only'] / total if total > 0 else 0
    summary['pct_cat_only'] = gene_counts['cat_only'] / total if total > 0 else 0
    summary.to_csv(SUMMARY_DIR / 'gene_presence_summary.tsv', sep='\t', index=False)
    
    # Save top missing genes
    pd.DataFrame([
        {'gene': k, 'n_assemblies': v, 'missing_from': 'CAT'} 
        for k, v in sorted(ensembl_missing_genes.items(), key=lambda x: -x[1])[:50]
    ] + [
        {'gene': k, 'n_assemblies': v, 'missing_from': 'Ensembl'} 
        for k, v in sorted(cat_missing_genes.items(), key=lambda x: -x[1])[:50]
    ]).to_csv(SUMMARY_DIR / 'top_missing_genes.tsv', sep='\t', index=False)
    
    print(f"Saved gene presence stats: {total} total occurrences")
    
    del ensembl_missing_genes, cat_missing_genes
    gc.collect()
    
    return summary

presence_summary = process_gene_presence_chunked()
print("\n✓ Gene presence processing complete")

## 4. Process Multi-Mapping (Lightweight - just count)

In [ ]:
def process_multi_mapping_chunked():
    """Process multi-mapping and save counts & classifications."""
    files = list(QC_DIR.rglob('*_multi_mapping.tsv'))
    print(f"Found {len(files)} multi-mapping files")
    
    if not files:
        return None
    
    ensembl_counts = []
    cat_counts = []
    ensembl_classifications = {}
    cat_classifications = {}
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                ensembl_multi = df[df['source'] == 'ensembl']
                cat_multi = df[df['source'] == 'cat']
                
                if len(ensembl_multi) > 0:
                    ensembl_counts.extend(ensembl_multi['n_matches'].tolist())
                    if 'classification' in ensembl_multi.columns:
                        for cls, count in ensembl_multi['classification'].value_counts().items():
                            ensembl_classifications[cls] = ensembl_classifications.get(cls, 0) + count
                            
                if len(cat_multi) > 0:
                    cat_counts.extend(cat_multi['n_matches'].tolist())
                    if 'classification' in cat_multi.columns:
                        for cls, count in cat_multi['classification'].value_counts().items():
                            cat_classifications[cls] = cat_classifications.get(cls, 0) + count
                    
            except Exception as e:
                print(f"Error loading {f}: {e}")
    
    # Save summary
    summary = pd.DataFrame([{
        'ensembl_genes_multi_mapped': len(ensembl_counts),
        'cat_genes_multi_mapped': len(cat_counts),
        'mean_ensembl_matches': np.mean(ensembl_counts) if ensembl_counts else 0,
        'mean_cat_matches': np.mean(cat_counts) if cat_counts else 0,
    }])
    summary.to_csv(SUMMARY_DIR / 'multi_mapping_summary.tsv', sep='\t', index=False)
    
    # Save classifications
    pd.DataFrame([
        {'source': 'ensembl', 'classification': k, 'count': v} for k, v in ensembl_classifications.items()
    ] + [
        {'source': 'cat', 'classification': k, 'count': v} for k, v in cat_classifications.items()
    ]).to_csv(SUMMARY_DIR / 'multi_mapping_classifications.tsv', sep='\t', index=False)
    
    print(f"Saved multi-mapping stats: {len(ensembl_counts)} Ensembl, {len(cat_counts)} CAT")
    
    return summary

multi_summary = process_multi_mapping_chunked()
print("\n✓ Multi-mapping processing complete")

## 5. Process RBH Pairs (Sample for visualization)

In [ ]:
def process_rbh_pairs_chunked():
    """Process RBH pairs and save statistics + sample for plotting."""
    files = list(RESULTS_DIR.rglob('*.gene_pairs_rbh.tsv'))
    print(f"Found {len(files)} RBH files")
    
    if not files:
        return None
    
    total_pairs = 0
    high_overlap_count = 0
    classification_counts = {}
    coverage_sample = []
    SAMPLE_SIZE = 10000  # Sample for scatter plot
    
    for i in range(0, len(files), CHUNK_SIZE):
        chunk_files = files[i:i+CHUNK_SIZE]
        print(f"Processing chunk {i//CHUNK_SIZE + 1}/{(len(files)-1)//CHUNK_SIZE + 1}")
        
        chunk_dfs = []
        for f in chunk_files:
            try:
                df = pd.read_csv(f, sep='\t')
                chunk_dfs.append(df)
            except Exception as e:
                print(f"Error loading {f}: {e}")
        
        if chunk_dfs:
            chunk_data = pd.concat(chunk_dfs, ignore_index=True)
            
            total_pairs += len(chunk_data)
            high_overlap_count += ((chunk_data['frac_ensembl_covered'] >= 0.9) & 
                                   (chunk_data['frac_cat_covered'] >= 0.9)).sum()
            
            # Classification counts
            if 'classification' in chunk_data.columns:
                for cls, count in chunk_data['classification'].value_counts().items():
                    classification_counts[cls] = classification_counts.get(cls, 0) + count
            
            # Sample for scatter plot
            if len(coverage_sample) < SAMPLE_SIZE:
                sample = chunk_data[['frac_ensembl_covered', 'frac_cat_covered']].sample(
                    min(SAMPLE_SIZE - len(coverage_sample), len(chunk_data))
                )
                coverage_sample.append(sample)
            
            del chunk_data, chunk_dfs
            gc.collect()
    
    # Save summary
    summary = pd.DataFrame([{
        'total_rbh_pairs': total_pairs,
        'high_overlap_pairs': high_overlap_count,
        'pct_high_overlap': high_overlap_count / total_pairs if total_pairs > 0 else 0
    }])
    summary.to_csv(SUMMARY_DIR / 'rbh_pairs_summary.tsv', sep='\t', index=False)
    
    # Save classification breakdown
    pd.DataFrame([
        {'classification': k, 'count': v, 'percentage': v/total_pairs if total_pairs > 0 else 0}
        for k, v in classification_counts.items()
    ]).to_csv(SUMMARY_DIR / 'rbh_classifications.tsv', sep='\t', index=False)
    
    # Save coverage sample
    if coverage_sample:
        sample_df = pd.concat(coverage_sample, ignore_index=True)
        sample_df.to_csv(SUMMARY_DIR / 'rbh_coverage_sample.tsv', sep='\t', index=False)
    
    print(f"Saved RBH stats: {total_pairs} total pairs, {len(coverage_sample)} in sample")
    
    del coverage_sample, classification_counts
    gc.collect()
    
    return summary

rbh_summary = process_rbh_pairs_chunked()
print("\n✓ RBH pairs processing complete")

## 6. Generate Visualizations from Saved Data

Now load only the small summary files and generate plots.

In [ ]:
print("\n" + "="*80)
print("GENERATING SUMMARY REPORT")
print("="*80 + "\n")

# Load summary data (small files)
concordance_rates = pd.read_csv(SUMMARY_DIR / 'transcript_concordance_rates.tsv', sep='\t')
coding_overall = pd.read_csv(SUMMARY_DIR / 'coding_integrity_overall.tsv', sep='\t')
presence_summary = pd.read_csv(SUMMARY_DIR / 'gene_presence_summary.tsv', sep='\t')
multi_summary = pd.read_csv(SUMMARY_DIR / 'multi_mapping_summary.tsv', sep='\t')
rbh_summary = pd.read_csv(SUMMARY_DIR / 'rbh_pairs_summary.tsv', sep='\t')

print("Summary Statistics:")
print(f"  Total RBH pairs analyzed: {rbh_summary['total_rbh_pairs'].iloc[0]:,}")
print(f"  Gene pairs with transcript data: {len(concordance_rates):,}")
print(f"  Mean transcript concordance: {concordance_rates['concordance_rate'].mean():.2%}")
print(f"  Protein-coding genes with CDS: {coding_overall['genes_with_cds_both'].iloc[0]:,}")
print(f"  Start & stop codon agreement: {coding_overall['pct_both_match'].iloc[0]:.1%}")
print(f"  Multi-mapped Ensembl genes: {multi_summary['ensembl_genes_multi_mapped'].iloc[0]:,}")
print(f"  Multi-mapped CAT genes: {multi_summary['cat_genes_multi_mapped'].iloc[0]:,}")

### 6.1 Transcript Concordance Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Breakdown of Match Types (Overall)
# We need to sum the counts from the per-assembly stats
per_assembly = pd.read_csv(SUMMARY_DIR / 'transcript_concordance_per_assembly.tsv', sep='\t')
cols = ['n_exact_matches', 'n_intron_matches', 'n_subset', 'n_superset', 
        'n_partial_5', 'n_partial_3', 'n_other_partial', 'n_unmatched']
counts = [per_assembly[c].sum() for c in cols if c in per_assembly.columns]
labels = [c.replace('n_', '').replace('_', ' ').title() for c in cols if c in per_assembly.columns]

total = sum(counts)
pcts = [c/total*100 for c in counts]

axes[0].barh(labels, pcts, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Percentage of Transcripts')
axes[0].set_title(f'Transcript Structural Concordance\n({total:,} total transcripts)')
for i, v in enumerate(pcts):
    axes[0].text(v, i, f' {v:.1f}%', va='center')

# Plot 2: Jaccard Index Distribution
if (SUMMARY_DIR / 'transcript_jaccard_indices.tsv').exists():
    jaccard = pd.read_csv(SUMMARY_DIR / 'transcript_jaccard_indices.tsv', sep='\t')
    axes[1].hist(jaccard['jaccard_index'], bins=50, edgecolor='black', color='lightgreen')
    axes[1].set_xlabel('Exon Jaccard Index')
    axes[1].set_ylabel('Number of Gene Pairs')
    axes[1].set_title('Distribution of Exon Overlap (Jaccard)')
    axes[1].set_yscale('log')
else:
    axes[1].text(0.5, 0.5, 'Jaccard data not available', ha='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'transcript_concordance_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved transcript_concordance_detailed.png")

### 6.2 Coding Integrity Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Classification Breakdown
if (SUMMARY_DIR / 'coding_classifications.tsv').exists():
    classifications = pd.read_csv(SUMMARY_DIR / 'coding_classifications.tsv', sep='\t')
    classifications = classifications.sort_values('count', ascending=True)
    
    axes[0].barh(classifications['classification'], classifications['percentage']*100, color='coral', edgecolor='black')
    axes[0].set_xlabel('Percentage of Genes')
    axes[0].set_title('Coding Integrity Classification')
    for i, row in classifications.iterrows():
        idx = list(classifications.index).index(i)
        axes[0].text(row['percentage']*100, idx, f" {row['percentage']*100:.1f}%", va='center')
else:
    axes[0].text(0.5, 0.5, 'Classification data not available', ha='center')

# Plot 2: CDS length differences (Log Scale)
length_diffs = pd.read_csv(SUMMARY_DIR / 'cds_length_differences_sample.tsv', sep='\t')
axes[1].hist(length_diffs['length_difference'], bins=50, edgecolor='black', log=True)
axes[1].set_xlabel('Absolute CDS Length Difference (bp)')
axes[1].set_ylabel('Number of Genes (Log Scale)')
axes[1].set_title(f'CDS Length Differences\n(Sample of {len(length_diffs):,} genes)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coding_integrity_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved coding_integrity_detailed.png")

### 6.3 Gene Presence/Absence & Multi-Mapping

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Gene presence/absence (Keep existing)
presence = presence_summary.iloc[0]
categories = ['Both\nAnnotations', 'Ensembl\nOnly', 'CAT\nOnly']
counts = [presence['both'], presence['ensembl_only'], presence['cat_only']]
colors = ['green', 'blue', 'orange']
axes[0].bar(categories, counts, color=colors, edgecolor='black', alpha=0.7)
axes[0].set_title('Gene Presence/Absence')

# Plot 2: Multi-Mapping Classifications
if (SUMMARY_DIR / 'multi_mapping_classifications.tsv').exists():
    mm_class = pd.read_csv(SUMMARY_DIR / 'multi_mapping_classifications.tsv', sep='\t')
    
    # Prepare data for stacked bar
    sources = ['ensembl', 'cat']
    # aggregated types: Split, Merge, Duplication, Collapse, Fragmented
    
    # Filter to known types
    known_types = ['Split', 'Merge', 'Duplication', 'Collapse', 'Fragmented']
    
    # Plot side-by-side or stacked? Let's do simple bar chart of types
    sns.barplot(data=mm_class, x='classification', y='count', hue='source', ax=axes[1])
    axes[1].set_title('Multi-Mapping Classifications')
    axes[1].set_yscale('log')
    axes[1].set_ylabel('Count (Log Scale)')
else:
    axes[1].text(0.5, 0.5, 'Multi-mapping data not available', ha='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'presence_multimapping_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved presence_multimapping_detailed.png")

### 6.4 RBH Overlap Quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Reciprocal coverage scatter (from sample)
coverage_sample = pd.read_csv(SUMMARY_DIR / 'rbh_coverage_sample.tsv', sep='\t')
axes[0].scatter(coverage_sample['frac_ensembl_covered'], 
               coverage_sample['frac_cat_covered'], alpha=0.3, s=10)
axes[0].axhline(0.9, color='red', linestyle='--', alpha=0.5, label='90% threshold')
axes[0].axvline(0.9, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Fraction of Ensembl Gene Covered')
axes[0].set_ylabel('Fraction of CAT Gene Covered')
axes[0].set_title(f'Reciprocal Coverage in RBH Pairs\n(Sample of {len(coverage_sample):,} pairs)')
axes[0].legend()
axes[0].set_xlim([0, 1.05])
axes[0].set_ylim([0, 1.05])

# Plot 2: Classification breakdown
classifications = pd.read_csv(SUMMARY_DIR / 'rbh_classifications.tsv', sep='\t')
classifications = classifications.sort_values('count', ascending=True)
axes[1].barh(range(len(classifications)), classifications['count'], edgecolor='black')
axes[1].set_yticks(range(len(classifications)))
axes[1].set_yticklabels(classifications['classification'])
axes[1].set_xlabel('Number of Gene Pairs')
axes[1].set_title('RBH Pair Overlap Classification')
for i, row in classifications.iterrows():
    idx = list(classifications.index).index(i)
    axes[1].text(row['count'], idx, f" {int(row['count']):,} ({row['percentage']*100:.1f}%)", va='center')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rbh_overlap_quality.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved rbh_overlap_quality.png")

## 7. Generate Final Summary Table

In [ ]:
# Create comprehensive summary table
summary_table = pd.DataFrame([
    ['Total RBH Pairs', f"{rbh_summary['total_rbh_pairs'].iloc[0]:,}", 
     f"{rbh_summary['pct_high_overlap'].iloc[0]:.1%}", '≥90% reciprocal overlap'],
    ['Transcript Concordance', f"{concordance_rates['concordance_rate'].mean():.1%}",
     f"{(concordance_rates['concordance_rate'] == 1.0).sum():,} / {len(concordance_rates):,}",
     'Mean rate; perfect matches'],
    ['Start & Stop Codons Match', f"{coding_overall['pct_both_match'].iloc[0]:.1%}",
     f"{int(coding_overall['both_codons_match'].iloc[0]):,} / {int(coding_overall['genes_with_cds_both'].iloc[0]):,}",
     'Protein-coding genes with CDS'],
    ['Potential Frameshifts', f"{coding_overall['pct_frameshift'].iloc[0]:.1%}",
     f"{int(coding_overall['frameshifts_detected'].iloc[0]):,}",
     'Non-divisible-by-3 length diffs'],
    ['Genes in Both Annotations', f"{presence_summary['pct_both'].iloc[0]:.1%}",
     f"{int(presence_summary['both'].iloc[0]):,} / {int(presence_summary['total'].iloc[0]):,}",
     'Named genes across all assemblies'],
    ['Multi-mapped Genes', f"{int(multi_summary['ensembl_genes_multi_mapped'].iloc[0] + multi_summary['cat_genes_multi_mapped'].iloc[0]):,}",
     f"E: {int(multi_summary['ensembl_genes_multi_mapped'].iloc[0]):,}, C: {int(multi_summary['cat_genes_multi_mapped'].iloc[0]):,}",
     '1-to-many or many-to-1']
], columns=['Metric', 'Value', 'Count', 'Description'])

print("\n" + "="*100)
print("FINAL QC SUMMARY TABLE")
print("="*100)
print(summary_table.to_string(index=False))
print("="*100)

summary_table.to_csv(OUTPUT_DIR / 'qc_summary_table.tsv', sep='\t', index=False)
print(f"\n✓ Summary table saved to: {OUTPUT_DIR / 'qc_summary_table.tsv'}")

## 8. Per-Assembly Summary for Downstream Analysis

In [ ]:
# Merge per-assembly summaries
transcript_per_asm = pd.read_csv(SUMMARY_DIR / 'transcript_concordance_per_assembly.tsv', sep='\t')
coding_per_asm = pd.read_csv(SUMMARY_DIR / 'coding_integrity_per_assembly.tsv', sep='\t')

per_assembly_final = transcript_per_asm.merge(
    coding_per_asm, on='assembly_accession', how='outer'
)

per_assembly_final.to_csv(OUTPUT_DIR / 'per_assembly_qc_summary.tsv', sep='\t', index=False)
print(f"✓ Per-assembly summary saved: {len(per_assembly_final)} assemblies")
print(f"  File: {OUTPUT_DIR / 'per_assembly_qc_summary.tsv'}")

## Summary

### Generated Files:
1. **Plots** (PNG):
   - `transcript_concordance_summary.png`
   - `coding_integrity_summary.png`
   - `presence_multimapping_summary.png`
   - `rbh_overlap_quality.png`

2. **Summary Tables** (TSV):
   - `qc_summary_table.tsv` - Overall metrics
   - `per_assembly_qc_summary.tsv` - Per-assembly breakdown

3. **Intermediate Data** (in `summary_stats/`):
   - All detailed statistics saved for further analysis

### Memory Usage:
- Processed 400+ assemblies in chunks
- Never loaded full dataset into memory
- All intermediate results saved to disk

### Next Steps:
1. Investigate assemblies with low concordance
2. Examine genes with coding integrity issues
3. Analyze annotation-specific genes
4. Population-level analysis using per-assembly summaries